# Movie Genre Classification System

End-to-end NLP workflow using the IMDb Genre Classification Kaggle dataset. This notebook mirrors the production code in `src/` for EDA, preprocessing, TF-IDF vectorization, model training, evaluation, and artifact saving.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from preprocess import load_dataset, clean_dataframe

## Load Dataset

Place the Kaggle file at `dataset/train_data.txt`. If it is not present, the project sample dataset is used for a quick smoke test.

In [ ]:
dataset_path = PROJECT_ROOT / 'dataset' / 'train_data.txt'
if not dataset_path.exists():
    dataset_path = PROJECT_ROOT / 'dataset' / 'sample_movies.csv'

df = load_dataset(dataset_path)
df.head()

In [ ]:
df.info()
df['genre'].value_counts().head(20)

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(12, 7))
order = df['genre'].value_counts().index
sns.countplot(data=df, y='genre', order=order, palette='viridis', hue='genre', legend=False)
plt.title('Genre Distribution')
plt.xlabel('Number of Movies')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

## Text Cleaning

The preprocessing module lowercases text, removes punctuation and numbers, tokenizes words, and removes English stopwords.

In [ ]:
clean_df = clean_dataframe(df)
clean_df[['genre', 'description', 'clean_description']].head()

## TF-IDF Vectorization and Train-Test Split

In [ ]:
stratify = clean_df['genre'] if clean_df['genre'].value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    clean_df['clean_description'], clean_df['genre'], test_size=0.2, random_state=42, stratify=stratify
)

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=1)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
X_train_tfidf.shape

## Train and Compare Models

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, n_jobs=-1, C=4.0),
    'Linear SVM': LinearSVC(C=1.0),
}

results = {}
predictions = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    results[name] = accuracy_score(y_test, preds)
    predictions[name] = preds

pd.DataFrame({'model': results.keys(), 'accuracy': results.values()}).sort_values('accuracy', ascending=False)

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x=list(results.keys()), y=list(results.values()), palette='mako', hue=list(results.keys()), legend=False)
plt.ylim(0, 1)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.tight_layout()
plt.show()

## Evaluate Best Model

In [ ]:
best_model_name = max(results, key=results.get)
best_predictions = predictions[best_model_name]
print(best_model_name)
print(classification_report(y_test, best_predictions, zero_division=0))

In [ ]:
labels = sorted(clean_df['genre'].unique())
cm = confusion_matrix(y_test, best_predictions, labels=labels)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted Genre')
plt.ylabel('Actual Genre')
plt.tight_layout()
plt.show()

## Production Training

Run the project training script to save the best model, vectorizer, metrics, and charts.

In [ ]:
%run ../src/train_model.py